# Waveform Acquisition from Tektronix DPO 2014B using `tm_devices`

This notebook connects to a Tektronix DPO 2014B oscilloscope over Ethernet using `tm_devices` with the pure-Python `pyvisa-py` backend, retrieves active channel waveforms, saves the data to a CSV file, and plots them inline.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tm_devices import DeviceManager
from tm_devices.helpers import PYVISA_PY_BACKEND

# Enable inline plotting
%matplotlib inline

## 1. Connect to Oscilloscope

Use the `DeviceManager` from `tm_devices` to handle connection lifetime. We explicitly use the pure-Python `@py` backend (`PYVISA_PY_BACKEND`) and define a compatibility-friendly VXI-11 resource string `TCPIP::<IP>::INSTR`.

In [ ]:
IP_ADDRESS = "10.10.10.2"
resource_expression = f"TCPIP::{IP_ADDRESS}::INSTR"

print(f"Connecting to oscilloscope at {IP_ADDRESS} using tm_devices...")
dm = DeviceManager()
dm.visa_library = PYVISA_PY_BACKEND

try:
    scope = dm.add_unsupported_device(resource_expression)
    idn = scope.query("*IDN?")
    print(f"Connected: {idn.strip()}")
except Exception as e:
    print(f"Connection failed: {e}")

## 2. Detect Active Channels

Queries the oscilloscope to see which analog channels (CH1-CH4) are currently turned on.

In [ ]:
active_channels = []
for ch in ["CH1", "CH2", "CH3", "CH4"]:
    is_active = scope.query(f"SELect:{ch}?").strip()
    if is_active == "1":
        active_channels.append(ch)

print(f"Active channels detected: {active_channels}")

## 3. Acquire Waveform Data

For each active channel, we:
1. Configure the data source and formatting (16-bit signed binary).
2. Retrieve vertical scaling (multiplier, offset) and horizontal scaling (sampling interval) parameters.
3. Download and parse the raw binary curve using `query_binary_values` from the `tm_devices` API (which wraps PyVISA to automatically decode the IEEE 488.2 block header and output a NumPy array).
4. Scale the raw integers into physical voltages.

In [ ]:
# Set data format parameters
scope.write("DATa:ENCdg RIBinary")  # Signed integer binary format
scope.write("DATa:WIDth 2")        # 16-bit resolution

waveforms = {}
time_axis = None
yunit = "V"
xunit = "s"

for ch in active_channels:
    print(f"Acquiring {ch}...")
    scope.write(f"DATa:SOUrce {ch}")
    
    # Get record length
    record_length = int(scope.query("HORizontal:RECOrdlength?").strip())
    scope.write("DATa:STARt 1")
    scope.write(f"DATa:STOP {record_length}")
    
    # Get scaling parameters
    xincr = float(scope.query("WFMPre:XINcr?").strip())
    xzero = float(scope.query("WFMPre:XZEro?").strip())
    pt_off = float(scope.query("WFMPre:PT_Off?").strip())
    ymult = float(scope.query("WFMPre:YMUlt?").strip())
    yoff = float(scope.query("WFMPre:YOFf?").strip())
    yzero = float(scope.query("WFMPre:YZEro?").strip())
    yunit = scope.query("WFMPre:YUNit?").strip().replace('"', '')
    xunit = scope.query("WFMPre:XUNit?").strip().replace('"', '')
    
    if xunit == "V" or not xunit:
        xunit = "s"
    
    # Download and parse curve directly into a NumPy array
    raw_ints = scope.query_binary_values(
        "CURVe?",
        datatype="h",         # 'h' = signed 16-bit short
        is_big_endian=True,   # RIBinary with 2 bytes is big-endian
        container=np.ndarray
    )
    
    # Scale data to physical units
    scaled_data = (raw_ints - yoff) * ymult + yzero
    waveforms[ch] = scaled_data
    
    # Create time axis on first channel run
    if time_axis is None:
        time_axis = xzero + xincr * (np.arange(len(raw_ints)) - pt_off)

print("Waveforms acquired successfully!")

## 4. Plot Waveforms

Generates a plot of the acquired channels with appropriate time scaling (e.g., ms, µs, ns).

In [ ]:
plt.figure(figsize=(12, 6))

# Determine readable time scale units
max_time = np.max(np.abs(time_axis))
time_multiplier = 1.0
plot_time_unit = xunit

if max_time < 1e-6:
    time_multiplier = 1e9
    plot_time_unit = "ns"
elif max_time < 1e-3:
    time_multiplier = 1e6
    plot_time_unit = "µs"
elif max_time < 1.0:
    time_multiplier = 1e3
    plot_time_unit = "ms"

for ch, data in waveforms.items():
    plt.plot(time_axis * time_multiplier, data, label=ch)

plt.title(f"Waveform Acquisition from DPO 2014B ({idn.strip()})")
plt.xlabel(f"Time ({plot_time_unit})")
plt.ylabel(f"Amplitude ({yunit})")
plt.grid(True)
plt.legend()
plt.show()

## 5. Export to CSV

Exports the time and voltage arrays to a CSV file.

In [ ]:
csv_filename = "waveform_data.csv"
csv_header = f"Time ({xunit})," + ",".join([f"{ch} ({yunit})" for ch in waveforms.keys()])
csv_data = np.column_stack([time_axis] + [waveforms[ch] for ch in waveforms.keys()])
np.savetxt(csv_filename, csv_data, delimiter=",", header=csv_header, comments="")
print(f"Saved data to {csv_filename}")

## 6. Disconnect

Close the connection to the oscilloscope and clean up the DeviceManager.

In [ ]:
dm.close()
print("DeviceManager closed. Connection ended.")